In [ ]:
# ============================================================
# Experiment 2 – Eye-Tracking Preprocessing
# ============================================================

# Description:
# Preprocessing of eye-tracking Time to First Fixation (TFF)
# data for the explicit dual-picture AAT.
#
# Steps:
# 1. Merge session data
# 2. Remove TFFs < 100 ms
# 3. Trim TFFs per session using ±2 SD
# 4. Export cleaned eye-tracking dataset
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# Parameters
# ============================================================

TFF_MIN = 100
TFF_TRIM_SD = 2

SESSION_LENGTH = 88
EXPECTED_TRIALS = 176

participants_to_exclude = [8, 29, 43, 44, 48, 64, 71, 72]

# ============================================================
# Helper Functions
# ============================================================

def load_csv(name):

    return pd.read_csv(
        name,
        header=None
    ).to_numpy(dtype=float)


def merge_sessions(arr):

    merged = []

    for p in range(arr.shape[1] // 2):

        merged.append(
            np.concatenate([
                arr[:, 2*p],
                arr[:, 2*p + 1]
            ])
        )

    return merged


def trim_sd(x, sd=2):

    x = np.asarray(x, dtype=float)

    out = x.copy()

    valid = np.isfinite(out)

    if valid.sum() < 3:
        return out

    m = np.nanmean(out[valid])
    s = np.nanstd(out[valid], ddof=1)

    lo = m - sd * s
    hi = m + sd * s

    out[
        np.isfinite(out)
        & ((out < lo) | (out > hi))
    ] = np.nan

    return out

# ============================================================
# Load Raw Data
# ============================================================

tff_raw = load_csv(
    "Raw_Data/timeFirstFixation.csv"
)

# ============================================================
# Merge Sessions
# ============================================================

tff_parts = merge_sessions(tff_raw)

# ============================================================
# Preprocess TFF
# ============================================================

rows = []

for pid in range(1, len(tff_parts) + 1):

    if pid in participants_to_exclude:
        continue

    tff = tff_parts[pid - 1].copy()

    # --------------------------------------------------------
    # Remove anticipatory first fixations
    # --------------------------------------------------------

    tff[
        np.isfinite(tff)
        & (tff < TFF_MIN)
    ] = np.nan

    # --------------------------------------------------------
    # Session-wise trimming
    # --------------------------------------------------------

    s1 = trim_sd(
        tff[:SESSION_LENGTH],
        sd=TFF_TRIM_SD
    )

    s2 = trim_sd(
        tff[SESSION_LENGTH:],
        sd=TFF_TRIM_SD
    )

    tff_clean = np.concatenate([
        s1,
        s2
    ])

    valid = np.isfinite(tff_clean)

    for t in np.where(valid)[0]:

        rows.append({

            "participant": pid,

            "trial": int(t + 1),

            "TFF_ms": float(tff_clean[t]),

            "logTFF": float(
                np.log10(tff_clean[t])
            )
        })

# ============================================================
# Save
# ============================================================

df_et = pd.DataFrame(rows)

df_et.to_csv(
    "Processed_Data/Exp2_ET_clean.csv",
    index=False
)

print("Eye-tracking preprocessing complete.")
print(f"Final valid TFF trials: {len(df_et)}")

In [ ]:
# ============================================================
# Experiment 2 – Create Final Analysis Dataset
# ============================================================

# Description:
# Merges cleaned behavioural and eye-tracking data for the
# explicit dual-picture AAT and creates the effect-coded dataset
# used for statistical analyses.
# ============================================================

import pandas as pd

# ============================================================
# Load Cleaned Data
# ============================================================

beh = pd.read_csv(
    "Processed_Data/Exp2_behavioural_clean.csv"
)

et = pd.read_csv(
    "Processed_Data/Exp2_ET_clean.csv"
)

# ============================================================
# Merge Behavioural and Eye-Tracking Data
# ============================================================

df = beh.merge(
    et,
    on=["participant", "trial"],
    how="inner"
)

# ============================================================
# Effect Coding
# ============================================================

df["side"] = df["side"].map({
    "left": -1,
    "right": 1
})

df["valence"] = df["valence"].map({
    "negative": -1,
    "positive": 1
})

df["congruence"] = df["congruence"].map({
    "incongruent": -1,
    "congruent": 1
})

# ============================================================
# Trial Index
# ============================================================

df["trial_index"] = (
    df.groupby("participant")
    .cumcount() + 1
)

# ============================================================
# Save
# ============================================================

df.to_csv(
    "Processed_Data/Exp2_ready.csv",
    index=False
)

print("Final analysis dataset saved.")
print(f"Final merged trials: {len(df)}")